# Notebook 1 of 2 — Ingest: Internet → ADLS → Bronze

**Your job:** get NYC taxi and weather data from the internet into your lake, keep an untouched copy, then store it as **Bronze Delta** and register it in Unity Catalog.

> **Learning rule:** AI tools are allowed here. Understanding is not optional. You will explain every transformation in your own words in `OBSERVATION_REPORT.md`.

### The naming contract (do not improvise)

| Layer | ADLS path (inside container `lakehouse`) | Unity Catalog table |
|---|---|---|
| Raw taxi files | `raw/taxi/yellow_tripdata_2025-08.parquet` (also `-09`, `-10`) | — |
| Raw weather file | `raw/weather/nyc_daily_2025-08-01_2025-10-31.json` | — |
| Bronze taxi | `bronze/taxi_trips/` | `<catalog>.bronze.taxi_trips` |
| Bronze weather | `bronze/weather_daily/` | `<catalog>.bronze.weather_daily` |

All tables are **external Delta tables** pointing at the same storage account and container.

### Setup
Fill in the two `<...>` values, then run the cell. Use the existing catalog you were given (`SHOW CATALOGS` lists it). You cannot create your own.

In [0]:
# ---- Fill in the two values marked <...> -------------------------------------
STORAGE_ACCOUNT = "stnycweatherdev001"   # your ADLS Gen2 account name
CATALOG         = "dbw_nyc_weather_dev"           # run SHOW CATALOGS and use the existing one given to you

# ---- Do NOT change these: the names are part of the contract ------------------
CONTAINER = "lakehouse"
BASE      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
MONTHS    = ["2025-08", "2025-09", "2025-10"]

TAXI_URL     = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
WEATHER_FILE = "nyc_daily_2025-08-01_2025-10-31.json"
WEATHER_URL  = ("https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.006"
                "&start_date=2025-08-01&end_date=2025-10-31"
                "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max"
                "&timezone=America%2FNew_York")

---
## Step 1 — Land the raw files in ADLS

**What:** download 3 monthly taxi Parquet files and 1 weather JSON, and put them **unchanged** in the `raw/` folders from the contract table.

**Why:** if your Bronze logic turns out wrong, you rebuild from your own copy instead of hitting the internet again. Raw is the evidence.

| Source | Where it comes from |
|---|---|
| Taxi (3 files, ~60–75 MB each) | `TAXI_URL` with each month from `MONTHS` |
| Weather (daily, NYC, Aug–Oct 2025) | `WEATHER_URL` (Open-Meteo, free, no key) |

**Hints**
- Spark cannot read an `https://` address. Python downloads it; Spark reads it afterwards.
- You are on serverless: download to the driver's local disk (`/tmp`) with `requests`, then move it into the lake with `dbutils.fs.cp("file:/tmp/...", "abfss://...")`.
- Stream the download in chunks (`stream=True`) instead of loading 70 MB into memory. Call `raise_for_status()` so a bad URL fails loudly.
- Delete the temp file afterwards. Use `dbutils.fs.ls` to prove the files arrived and note their sizes.

**Pause and observe →** Why store an untouched copy in `raw/` instead of writing Delta straight from the download? Answer in report §1.

In [0]:
import io
import pandas as pd
import requests

print("--- Landing Raw Taxi Files as Single Parquet Files via Spark & ADLS ---")
for month in MONTHS:
    url = TAXI_URL.format(month=month)
    final_adls_path = f"{BASE}/raw/taxi/yellow_tripdata_{month}.parquet"
    temp_adls_dir = f"{BASE}/raw/taxi_temp_{month}"
    
    print(f"Downloading {month} taxi data into memory...")
    response = requests.get(url)
    response.raise_for_status()
    
    # Load into Pandas, then Spark
    pdf = pd.read_parquet(io.BytesIO(response.content))
    sdf = spark.createDataFrame(pdf)
    
    print(f"Writing {month} data to temporary ADLS location...")
    # Coalesce to 1 to ensure a single partition file is generated
    sdf.coalesce(1).write.mode("overwrite").parquet(temp_adls_dir)
    
    print(f"Locating the generated part file for {month}...")
    files = dbutils.fs.ls(temp_adls_dir)
    part_file = [f.path for f in files if f.name.startswith("part-") and f.name.endswith(".parquet")][0]
    
    print(f"Moving part file to final path: {final_adls_path}")
    # Copy part file to final destination name
    dbutils.fs.cp(part_file, final_adls_path)
    
    print(f"Cleaning up temporary ADLS directory...")
    dbutils.fs.rm(temp_adls_dir, recurse=True)

print("\n--- Downloading and Landing Weather Data ---")
weather_adls_path = f"{BASE}/raw/weather/{WEATHER_FILE}"
print("Fetching weather JSON from Open-Meteo...")
weather_response = requests.get(WEATHER_URL)
weather_response.raise_for_status()

print(f"Writing weather data to ADLS: {weather_adls_path}")
dbutils.fs.put(weather_adls_path, weather_response.text, overwrite=True)

print("\n--- Verification: Listing files in raw/taxi/ ---")
display(dbutils.fs.ls(f"{BASE}/raw/taxi/"))


--- Landing Raw Taxi Files as Single Parquet Files via Spark & ADLS ---
Writing 2025-08 data to temporary ADLS location...
Locating the generated part file for 2025-08...
Moving part file to final path: abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/taxi/yellow_tripdata_2025-08.parquet
Cleaning up temporary ADLS directory...
Writing 2025-09 data to temporary ADLS location...
Locating the generated part file for 2025-09...
Moving part file to final path: abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/taxi/yellow_tripdata_2025-09.parquet
Cleaning up temporary ADLS directory...
Writing 2025-10 data to temporary ADLS location...
Locating the generated part file for 2025-10...
Moving part file to final path: abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/taxi/yellow_tripdata_2025-10.parquet
Cleaning up temporary ADLS directory...

--- Downloading and Landing Weather Data ---
Fetching weather JSON from Open-Meteo...
Writing weather data to ADLS: a

path,name,size,modificationTime
abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/taxi/yellow_tripdata_2025-08.parquet,yellow_tripdata_2025-08.parquet,78721710,1795021816000
abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/taxi/yellow_tripdata_2025-09.parquet,yellow_tripdata_2025-09.parquet,92737319,1795021847000
abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/taxi/yellow_tripdata_2025-10.parquet,yellow_tripdata_2025-10.parquet,96848146,1795021882000


---
## Step 2 — Bronze taxi: one Delta table for all three months

**Rules**
1. Read each month's raw Parquet and write to `bronze/taxi_trips/` as **Delta**.
2. Keep the **20 source columns exactly as they arrive** (names, types, values). No renaming, no casting, no filtering. Bronze is the evidence.
3. Add **three metadata columns**: `source_file` (string, the file name), `source_month` (string, `yyyy-MM`), `ingested_at` (timestamp).
4. Partition the table by `source_month`.
5. **Re-runnable:** running the cell twice must give the same row count, not double.

**Hints**
- Loop over `MONTHS`, one month at a time.
- `F.lit(...)` for constants, `F.current_timestamp()` for the load time.
- For rule 5 look at Delta's `replaceWhere` write option (with `mode("overwrite")`): it replaces only the month you are writing.
- Finish with `printSchema()` and a count per `source_month`.

**Pause and observe →** Which column types surprise you (look at `passenger_count`, the two timestamps, `RatecodeID`)? Why would a taxi meter count passengers in a `long`? Note row counts per month in report §2.

In [0]:
print("--- Building Bronze Taxi Table as an EXTERNAL Delta Table ---")

# Drop the existing managed table definition
spark.sql(f"DROP TABLE IF EXISTS bronze.taxi_trips")

bronze_taxi_path = f"{BASE}/bronze/taxi_trips"

for month in MONTHS:
    raw_path = f"{BASE}/raw/taxi/yellow_tripdata_{month}.parquet"
    
    print(f"Processing bronze ingestion for {month}...")
    
    # 1. Read raw Parquet file
    df_raw = spark.read.parquet(raw_path)
    
    # 2. Add required metadata audit columns
    df_bronze = (
        df_raw
        .withColumn("source_file", F.lit(f"yellow_tripdata_{month}.parquet"))
        .withColumn("source_month", F.lit(month))
        .withColumn("ingested_at", F.current_timestamp())
    )
    
    # 3. Write as an EXTERNAL Delta table using option("path", ...) and replaceWhere
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("path", bronze_taxi_path)
        .option("replaceWhere", f"source_month = '{month}'")
        .saveAsTable("bronze.taxi_trips")
    )

print("\n--- Bronze Table Schema & Verification ---")
spark.table("bronze.taxi_trips").printSchema()

print("\n--- Row Count per Source Month ---")
display(
    spark.table("bronze.taxi_trips")
    .groupBy("source_month")
    .count()
    .orderBy("source_month")
)

--- Building Bronze Taxi Table as an EXTERNAL Delta Table ---
Processing bronze ingestion for 2025-08...
Processing bronze ingestion for 2025-09...
Processing bronze ingestion for 2025-10...

--- Bronze Table Schema & Verification ---
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullabl

source_month,count
2025-08,3574091
2025-09,4251015
2025-10,4428699


---
## Step 3 — Bronze weather: flatten the JSON

**Rules**
1. The JSON contains one `daily` object made of **parallel arrays** (`time`, `temperature_2m_max`, …). Turn it into **one row per date**.
2. Keep the API's column names and values as they are. Add `source_file` and `ingested_at`.
3. Write Delta to `bronze/weather_daily/` (overwrite is fine; this one is a single file). Expect **92 rows**.

**Hints**
- JSON that spans many lines needs the `multiLine` read option.
- `select("daily.*")` gives you the arrays as columns. `F.arrays_zip` + `F.explode` turn parallel arrays into rows.

**Pause and observe →** Why is "one object with arrays" an awkward shape for analytics, and what does one row per date make easy? Report §2.

In [0]:
from pyspark.sql.functions import col, explode, arrays_zip, current_timestamp

weather_raw_path = f"{BASE}/raw/weather/{WEATHER_FILE}"
bronze_weather_path = f"{BASE}/bronze/weather_daily/"

print("Reading and flattening Open-Meteo weather JSON...")

# 1. Read JSON with multiLine enabled
df_raw = spark.read.option("multiLine", True).json(weather_raw_path)

# 2. Extract arrays from the daily struct and zip them together
df_daily = df_raw.select("daily.*")

df_bronze_weather = (
    df_daily
    .select(
        explode(
            arrays_zip(
                col("time"),
                col("temperature_2m_max"),
                col("temperature_2m_min"),
                col("precipitation_sum"),
                col("snowfall_sum"),
                col("wind_speed_10m_max")
            )
        ).alias("d")
    )
    .select(
        col("d.time").alias("time"),
        col("d.temperature_2m_max").alias("temperature_2m_max"),
        col("d.temperature_2m_min").alias("temperature_2m_min"),
        col("d.precipitation_sum").alias("precipitation_sum"),
        col("d.snowfall_sum").alias("snowfall_sum"),
        col("d.wind_speed_10m_max").alias("wind_speed_10m_max")
    )
    # 3. Add audit metadata columns
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp())
)

# 4. Write as Delta table
(
    df_bronze_weather.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(bronze_weather_path)
)

print("\n--- Bronze Weather Ingestion Complete ---")
df_result = spark.read.format("delta").load(bronze_weather_path)
print(f"Total rows (Expected: 92): {df_result.count()}")
display(df_result.limit(5))

Reading and flattening Open-Meteo weather JSON...

--- Bronze Weather Ingestion Complete ---
Total rows (Expected: 92): 92


time,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max,source_file,ingested_at
2025-08-01,23.0,17.8,1.0,0.0,16.1,abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/weather/nyc_daily_2025-08-01_2025-10-31.json,2026-11-17T22:23:39.751Z
2025-08-02,25.7,17.1,0.0,0.0,10.1,abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/weather/nyc_daily_2025-08-01_2025-10-31.json,2026-11-17T22:23:39.751Z
2025-08-03,28.1,16.5,0.0,0.0,11.1,abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/weather/nyc_daily_2025-08-01_2025-10-31.json,2026-11-17T22:23:39.751Z
2025-08-04,33.2,17.0,0.0,0.0,14.6,abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/weather/nyc_daily_2025-08-01_2025-10-31.json,2026-11-17T22:23:39.751Z
2025-08-05,31.0,19.9,0.0,0.0,9.7,abfss://lakehouse@stnycweatherdev001.dfs.core.windows.net/raw/weather/nyc_daily_2025-08-01_2025-10-31.json,2026-11-17T22:23:39.751Z


---
## Step 4 — Register the Bronze external tables

Create schema `bronze` in your catalog, then register `taxi_trips` and `weather_daily` as **external Delta tables** over the folders from Steps 2 and 3 (you already know how). Print `Type`, `Provider` and `Location` for each.

**Hint:** if `CREATE SCHEMA` complains about managed storage, tell your instructor. Do not work around it by writing data somewhere else.

**Pause and observe →** Paste the `DESCRIBE EXTENDED` evidence in report §4. If you ran `DROP TABLE` on an external table, what happens to the files in ADLS? Answer from understanding; **do not test it on your data.**

In [0]:
# 1. Create the bronze schema in your assigned Unity Catalog
print(f"Creating schema {CATALOG}.bronze...")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

# 2. Register taxi_trips external table
print("Registering bronze.taxi_trips...")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.taxi_trips
    USING DELTA
    LOCATION '{BASE}/bronze/taxi_trips/'
""")

# 3. Register weather_daily external table
print("Registering bronze.weather_daily...")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.weather_daily
    USING DELTA
    LOCATION '{BASE}/bronze/weather_daily/'
""")

print("\n--- Registration Complete! Inspecting Tables ---\n")

# 4. Display table descriptions and paths to verify type, provider, and location
print("--- Taxi Trips Table Details ---")
display(spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.bronze.taxi_trips"))

print("\n--- Weather Daily Table Details ---")
display(spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.bronze.weather_daily"))

Creating schema dbw_nyc_weather_dev.bronze...
Registering bronze.taxi_trips...
Registering bronze.weather_daily...

--- Registration Complete! Inspecting Tables ---

--- Taxi Trips Table Details ---


col_name,data_type,comment
VendorID,int,null
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
passenger_count,double,null
trip_distance,double,null
RatecodeID,double,null
store_and_fwd_flag,string,null
PULocationID,int,null
DOLocationID,int,null
payment_type,bigint,null



--- Weather Daily Table Details ---


col_name,data_type,comment
time,string,null
temperature_2m_max,double,null
temperature_2m_min,double,null
precipitation_sum,double,null
snowfall_sum,double,null
wind_speed_10m_max,double,null
source_file,string,null
ingested_at,timestamp,null
,,
# Delta Statistics Columns,,


---
## Step 5 — Validate, then prove it is re-runnable

Run the check cell below (it is given). Fix anything marked ❌. Then **re-run your Step 2 cell** and run the checks again. Record the counts before and after in report §3.

> **Git checkpoint:** export or commit this notebook now (`git commit -m "notebook 1: bronze ingestion"`).

In [0]:
from pyspark.sql import functions as F

_results = []
def check(label, ok):
    _results.append(bool(ok))
    print(("✅" if ok else "❌"), label)

def table_info(name):
    return {r.col_name: r.data_type for r in spark.sql(f"DESCRIBE EXTENDED {name}").collect()}

def summary():
    print(f"\n{sum(_results)} of {len(_results)} checks passed")

# ---------------- RAW ----------------
raw_taxi = sorted(f.name for f in dbutils.fs.ls(f"{BASE}/raw/taxi"))
check("raw/taxi holds exactly the 3 monthly parquet files",
      raw_taxi == [f"yellow_tripdata_{m}.parquet" for m in MONTHS])
check("raw/weather holds the weather JSON",
      [f.name for f in dbutils.fs.ls(f"{BASE}/raw/weather")] == [WEATHER_FILE])

# ---------------- BRONZE TAXI ----------------
t = f"{CATALOG}.bronze.taxi_trips"
info = table_info(t)
check("bronze.taxi_trips is an EXTERNAL table", info.get("Type") == "EXTERNAL")
check("bronze.taxi_trips points at <container>/bronze/taxi_trips", info.get("Location", "").rstrip("/").endswith("/bronze/taxi_trips"))
check("bronze.taxi_trips is Delta", str(info.get("Provider", "")).lower() == "delta")

bt = spark.table(t)
src_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
            "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "fare_amount",
            "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
            "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]
check("Bronze keeps the 20 source column names untouched, plus 3 metadata columns",
      set(bt.columns) == set(src_cols + ["source_file", "source_month", "ingested_at"]))

per_month = {r.source_month: r["count"] for r in bt.groupBy("source_month").count().collect()}
raw_counts = {m: spark.read.parquet(f"{BASE}/raw/taxi/yellow_tripdata_{m}.parquet").count() for m in MONTHS}
check("Bronze rows per month == rows in the raw file (no loss, no duplicates)", per_month == raw_counts)

# ---------------- BRONZE WEATHER ----------------
w = f"{CATALOG}.bronze.weather_daily"
winfo = table_info(w)
check("bronze.weather_daily is an EXTERNAL table at <container>/bronze/weather_daily",
      winfo.get("Type") == "EXTERNAL" and winfo.get("Location", "").rstrip("/").endswith("/bronze/weather_daily"))
bw = spark.table(w)
check("weather: 92 rows (Aug 1 - Oct 31) and 92 distinct dates", bw.count() == 92 and bw.select("time").distinct().count() == 92)
check("weather keeps the API column names",
      {"time", "temperature_2m_max", "temperature_2m_min", "precipitation_sum", "snowfall_sum",
       "wind_speed_10m_max", "source_file", "ingested_at"} == set(bw.columns))

summary()
print("Bronze rows per month:", dict(sorted(per_month.items())))

✅ raw/taxi holds exactly the 3 monthly parquet files
✅ raw/weather holds the weather JSON
✅ bronze.taxi_trips is an EXTERNAL table
✅ bronze.taxi_trips points at <container>/bronze/taxi_trips
✅ bronze.taxi_trips is Delta
✅ Bronze keeps the 20 source column names untouched, plus 3 metadata columns
✅ Bronze rows per month == rows in the raw file (no loss, no duplicates)
✅ bronze.weather_daily is an EXTERNAL table at <container>/bronze/weather_daily
✅ weather: 92 rows (Aug 1 - Oct 31) and 92 distinct dates
✅ weather keeps the API column names

10 of 10 checks passed
Bronze rows per month: {'2025-08': 3574091, '2025-09': 4251015, '2025-10': 4428699}
